# L3: Agent Loop

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In this lesson you'll see an agent work through a task one step at a time. The agent reviews a vague software requirement and produces a structured report that lists what is clear, what is ambiguous, and what needs to be asked before anyone starts building.

**By the end of this lesson you will be able to:**
- Describe the agent loop: the model decides, calls a tool, reads the result, and repeats until it is done
- Explain how an instruction file (`AGENTS.md`) steers the agent's behavior
- Tell custom tools apart from built-in file tools
- Read an execution trace and spot where the agent hit an error and recovered
- Explain why a turn limit (`MAX_TURNS`) is a useful safeguard

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

from deepagent_helpers import (WORKSPACE_DIR, ANALYSIS_PATH, 
    DEFAULT_TICKET_ID, langsmith_status, build_review_agent, get_ticket,
    run_review_loop_live, save_deepagent_trace, load_deepagent_trace)

from visualization_helpers import (render_review_trace, render_report, 
    render_requirement_comparison, plot_agent_tools)

%matplotlib inline
print("✓ Environment ready")

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"View"</em> option on the top menu of the notebook and then 2) click on <em>"File Browser"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>

## Requirements Review Agent

Our agent takes a raw requirement document and turns it into a structured review report, saved in a single file, `/analysis.md`.

The agent's behavior is defined in an instruction file called `AGENTS.md`. Print it below and read it carefully, because it is the agent's whole job description. Notice three things:
- **A fixed procedure:** six stages, in order, one tool call per turn
- **A report skeleton:** the exact sections the agent has to fill in
- **A rule for failures:** what to do when a tool call returns an error

In [ ]:
print(Path("workspace/memories/AGENTS.md").read_text())

##### Create the agent
Two settings matter here:
- `MODEL`: the language model that makes every decision in the loop
- `MAX_TURNS`: the maximum number of model turns allowed before the loop is stopped. Think of it as a safety cap, so a confused agent can't keep going forever.

In [ ]:
MODEL = "openai:gpt-5.4-mini"
MAX_TURNS = 12

agent = build_review_agent(
    model = MODEL,
    max_turns = MAX_TURNS,
)

print("Agent created")

| Tool | What it does |
|---|---|
| `fetch_requirement` | **Custom to this agent.** Fetches the requirement document. Simulates two transient failures before succeeding, so the Model has to retry. |
| `write_file` | Creates a new file, or overwrites one that already exists. Called once, to lay down the `analysis.md` skeleton. |
| `edit_file` | Makes an exact string replacement inside a file. Called once per remaining section, so each turn only touches the section it is filling in. |


## Execute the Agent Loop

Here is the loop you are about to run:

1. The model looks at the task and everything that has happened so far
2. It picks **one** tool to call (or decides it is finished)
3. The tool runs and returns a result, or an error
4. The result goes back to the model, and the cycle repeats

The loop ends when the model replies without calling a tool, or when `MAX_TURNS` is reached.

First, look at the input. The requirement below is deliberately vague. The agent's job is to find out *how* vague.

In [ ]:
TICKET_ID = DEFAULT_TICKET_ID
requirement_doc = get_ticket(TICKET_ID)

print(f"Ticket {TICKET_ID}:\n")
print(requirement_doc)

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> LLMs do not always produce the same results. Consequently, the agent behavior you experience when running labs may differ from the video.</p>

##### Run the agent

The next cell has a switch, `USE_LIVE_LLM`:
- `True`: runs the agent live against the model. The run takes a little time, and the exact steps may differ from run to run.
- `False`: loads a saved trace from a previous real run, so you can follow along without calling the model.

When the run finishes, the cell prints how many model turns it took and how many items the report contains in each section.

In [ ]:
USE_LIVE_LLM = True
if USE_LIVE_LLM:
    run = run_review_loop_live(agent, ticket_id=TICKET_ID)
    save_deepagent_trace(run, "lesson_3_live_trace.json")
    print(f"✓ Ran live against {MODEL}, saved to lesson_3_live_trace.json")
    print(langsmith_status())
else:
    run = load_deepagent_trace("lesson_3_prebaked_trace.json")
    print("✓ Loaded a pre-baked real trace")

print(f"✓ Completed in {run.turns} model turns ({len(run.trace)} steps)")
print(f" {ANALYSIS_PATH}: {run.counts}, summary {run.summary_words} words")

### Execution Trace

Click on the following link to view saved agent execution trace.

🔗   [[ Agent's Trace ]](https://smith.langchain.com/public/1f4c8774-8714-42b4-a37f-8740849d78ed/r/01a10fbe-c6c0-78d3-83e2-14060a8e2ac7?scroll_to=input&start_time=2026-10-06T05%3A45%3A23.39242Z)

The table below shows the same run in the notebook. Each row has an **Actor** (the model or a tool), an **Action**, and a **Result**. As you read it, look for:
- The order of the stages, and whether it matches the procedure in `AGENTS.md`
- Any **error** a tool returned, and what the model did next

In [ ]:
display(render_review_trace(run.trace))

## The Report

Finally, compare the original requirement (left) with the report the agent wrote (right). A good report should:
- Restate only what the document actually says (no inference)
- Quote each vague clause and explain why it is under-specified
- Ask the questions a developer would need answered before starting

In [ ]:
display(render_requirement_comparison(
    requirement_doc, run.analysis_markdown, report_title="analysis.md"))

## Conclusion

In this lesson you ran a complete agent loop from start to finish.

**Key takeaways**
- The model chooses an action, a tool executes it, the result returns to the model, and the cycle repeats until the model decides it is done.
- **Instructions shape behavior.** `AGENTS.md` set the procedure, the report format, and the retry rule, with no code changes needed.
- **Narrow tools are easier to control.** Each step used one small tool, which makes the agent's actions easy to follow and check.
- **Errors are information, not dead ends.** Because tool results flow back into the loop, the agent can see a failure, adjust, and try again. Look back at the trace for where this happened in your run.
- **Guardrails matter.** `MAX_TURNS` keeps a run bounded, and the trace gives you visibility into exactly what happened.
- **Runs vary.** The same agent can take different paths on different runs, which is why traces are valuable.

**Try it yourself**
- Change `TICKET_ID` or the requirement text and see how the report changes
- Lower `MAX_TURNS` (for example to 4) and see what happens when the agent runs out of turns before finishing